In [ ]:

import os
import yaml
import shutil
import datetime
from google.colab import drive

# --- CẤU HÌNH ---

MODEL_VERSION = 'n'
BATCH_SIZE = 64
EPOCHS = 30

# --- BƯỚC 1: KẾT NỐI DRIVE & TÌM DỮ LIỆU ---
print("🚀 Đang khởi động quy trình...")
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

# Tìm thư mục LP_detection trên Drive
possible_paths = [
    '/content/drive/MyDrive/License-Plate-Recognition-main/License-Plate-Recognition-main/LP_detection',
    '/content/drive/MyDrive/License-Plate-Recognition-main/LP_detection',
    '/content/drive/MyDrive/LP_detection'
]
DRIVE_LP_PATH = None
for path in possible_paths:
    if os.path.exists(path):
        DRIVE_LP_PATH = path
        print(f"✅ Đã tìm thấy dữ liệu Biển số tại: {DRIVE_LP_PATH}")
        break

if not DRIVE_LP_PATH:
    print("❌ LỖI: Không tìm thấy thư mục 'LP_detection' trên Drive này!")
    print("👉 Hãy upload folder LP_detection lên Drive của tài khoản này trước nhé.")
    raise FileNotFoundError("Chưa có dữ liệu LP_detection")

# Tạo nơi lưu model kết quả
FINAL_MODEL_DIR = os.path.join(os.path.dirname(DRIVE_LP_PATH), 'My_Models_Lightweight')
if not os.path.exists(FINAL_MODEL_DIR): os.makedirs(FINAL_MODEL_DIR)

# --- BƯỚC 2: COPY DỮ LIỆU VÀO MÁY ẢO (TĂNG TỐC) ---
LOCAL_DATA_DIR = '/content/temp_data/LP_detection'
if not os.path.exists(LOCAL_DATA_DIR):
    print(f"⏳ Đang copy dữ liệu xuống máy ảo (chờ xíu)...")
    shutil.copytree(DRIVE_LP_PATH, LOCAL_DATA_DIR)
    print("✅ Copy dữ liệu hoàn tất.")
else:
    print("✅ Dữ liệu máy ảo đã sẵn sàng.")

# --- BƯỚC 3: CÀI ĐẶT YOLOv5 MỚI NHẤT ---
if not os.path.exists('/content/yolov5'):
    print("⬇️ Đang tải YOLOv5...")
    !git clone https://github.com/ultralytics/yolov5 /content/yolov5
    %cd /content/yolov5
    !pip install -r requirements.txt -q
    !wandb disabled 2>/dev/null
else:
    %cd /content/yolov5
    print("✅ Môi trường YOLOv5 đã sẵn sàng.")

# --- BƯỚC 4: TẠO FILE CẤU HÌNH (1 CLASS: LICENSE_PLATE) ---
print("⚙️ Đang tạo file cấu hình...")
plate_yaml = {
    'train': f'{LOCAL_DATA_DIR}/images/train',
    'val': f'{LOCAL_DATA_DIR}/images/val',
    'nc': 1,                  # Chỉ có 1 class là biển số
    'names': ['license_plate']
}
with open('/content/plate_nano.yaml', 'w') as f: yaml.dump(plate_yaml, f)

# --- BƯỚC 5: TRAIN ---
WEIGHTS = f'yolov5{MODEL_VERSION}.pt'
print("\n" + "="*50)
print(f"🔥 BẮT ĐẦU TRAIN BIỂN SỐ: YOLOv5-{MODEL_VERSION.upper()}")
print("="*50)

!python train.py --img 640 \
                 --batch {BATCH_SIZE} \
                 --epochs {EPOCHS} \
                 --data /content/plate_nano.yaml \
                 --weights {WEIGHTS} \
                 --project /content/runs/train \
                 --name plate_nano_exp \
                 --exist-ok

# --- BƯỚC 6: LƯU TOÀN BỘ KẾT QUẢ (UPDATED) ---
print("\n💾 Đang lưu TOÀN BỘ kết quả (Model + Biểu đồ + Log) về Drive...")

# 1. Xác định thư mục nguồn (trong Colab)
source_dir = '/content/runs/train/plate_nano_exp'

# 2. Tạo tên thư mục đích có gắn ngày giờ (để không bị trùng)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
destination_dir = os.path.join(FINAL_MODEL_DIR, f'Plate_Result_{timestamp}_{MODEL_VERSION.upper()}')

# 3. Copy toàn bộ
try:
    if os.path.exists(source_dir):
        shutil.copytree(source_dir, destination_dir)
        print(f"🎉 THÀNH CÔNG RỰC RỠ! Đã lưu trọn bộ kết quả tại:")
        print(f"   📂 {destination_dir}")
        print("   (Trong folder này sẽ có: weights/best.pt, results.png, confusion_matrix...)")
    else:
        print("❌ Lỗi: Không tìm thấy thư mục kết quả trong Colab. Có thể quá trình train bị lỗi.")
except Exception as e:
    print(f"❌ Có lỗi khi copy sang Drive: {e}")

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
       6/29       8.9G    0.02655   0.006554          0        120        640:  61% 63/104 [01:08<00:44,  1.08s/it]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       6/29       8.9G    0.02656   0.006558          0        125        640:  62% 64/104 [01:09<00:38,  1.04it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       6/29       8.9G    0.02655   0.006538          0        104        640:  62% 65/104 [01:10<00:42,  1.10s/it]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
       6/29       8.9G    0.